# 🎯 Batch Scoring — Manufacturing Predictive Maintenance

Scores all equipment with the trained LightGBM model to produce
maintenance predictions and risk rankings.

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round, udf
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMClassificationModel

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load model and features
model = LightGBMClassificationModel.load('Files/models/predictive_maintenance_lgbm')
features_df = spark.read.table('gold_ml_features')
print(f'Scoring {features_df.count():,} feature rows')

In [ ]:
# Prepare features (same pipeline as training)
numeric_features = [
    'avg_temp', 'std_temp', 'max_temp', 'min_temp', 'temp_range',
    'avg_pressure', 'std_pressure', 'max_pressure',
    'avg_vibration', 'std_vibration', 'max_vibration',
    'avg_humidity',
    'reading_count', 'temp_anomaly_count', 'vib_anomaly_count', 'anomaly_ratio',
    'total_units', 'total_defects', 'total_downtime',
    'batch_count', 'avg_yield', 'avg_defect_rate',
    'equipment_age_days',
]

indexer_type = StringIndexer(inputCol='machine_type', outputCol='machine_type_idx', handleInvalid='keep')
indexer_line = StringIndexer(inputCol='production_line', outputCol='production_line_idx', handleInvalid='keep')
indexed_df = indexer_type.fit(features_df).transform(features_df)
indexed_df = indexer_line.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + ['machine_type_idx', 'production_line_idx']
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df)

In [ ]:
# Score all data
scored = model.transform(model_df)

# Extract probability of positive class (maintenance needed)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('maintenance_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_maintenance', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('maintenance_probability') > 0.8, 'critical')
        .when(col('maintenance_probability') > 0.6, 'high')
        .when(col('maintenance_probability') > 0.4, 'medium')
        .otherwise('low')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'machine_id', 'reading_date', 'production_line', 'machine_type',
        'avg_temp', 'avg_pressure', 'avg_vibration', 'avg_humidity',
        'total_downtime', 'total_defects', 'avg_defect_rate', 'avg_yield',
        'anomaly_ratio', 'equipment_age_days',
        'needs_maintenance', 'predicted_maintenance',
        'maintenance_probability', 'risk_level',
        'scored_at'
    )
)

predictions.write.mode('overwrite').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
print(f'\nRisk distribution:')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Create summary table — per machine risk ranking
summary = (
    predictions
    .groupBy('machine_id', 'production_line', 'machine_type')
    .agg(
        spark_round(avg('maintenance_probability'), 4).alias('avg_risk_score'),
        spark_sum('predicted_maintenance').alias('maintenance_predictions'),
        count('*').alias('total_observations'),
        spark_round(avg('total_downtime'), 1).alias('avg_daily_downtime'),
        spark_round(avg('avg_defect_rate'), 2).alias('avg_defect_rate'),
        spark_round(avg('anomaly_ratio'), 4).alias('avg_anomaly_ratio'),
    )
    .withColumn('maintenance_rate',
        spark_round(col('maintenance_predictions') / col('total_observations') * 100, 1)
    )
    .withColumn('overall_risk',
        when(col('avg_risk_score') > 0.6, 'high')
        .when(col('avg_risk_score') > 0.3, 'medium')
        .otherwise('low')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_risk_score').desc())
)

summary.write.mode('overwrite').format('delta').saveAsTable('gold_ml_summary')
print(f'\nMachine risk summary: {summary.count()} machines')
summary.show(10, truncate=False)

In [ ]:
# Optimize
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('\nAll Gold ML tables optimized')

print('\n=== Scoring Summary ===')
print(f'Total predictions: {predictions.count():,}')
print(f'Machines scored: {summary.count()}')
high_risk = summary.filter(col('overall_risk') == 'high').count()
print(f'High-risk machines: {high_risk}')
print('\nGold tables created:')
print('  - gold_ml_features')
print('  - gold_ml_model_metrics')
print('  - gold_ml_feature_importance')
print('  - gold_ml_predictions')
print('  - gold_ml_summary')